# 1 - Ground Truth Generation

To evaluate retrieval, we need to know which chunk a question *should* return.
We generate this ground truth with an LLM (module 4 approach): for a random
sample of chunks, the LLM formulates questions that this chunk answers. The
chunk the question was generated from is the "correct" retrieval result.

**Output:** `../data/ground-truth.csv` with columns `question, chunk_id, doc_id`
(committed to the repo so the evaluations are reproducible without re-paying
for generation).

Requires: `OPENAI_API_KEY` in `../.env`. Cost: ~150 small LLM calls.

In [1]:
import json
import random

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from tqdm.auto import tqdm

load_dotenv("../.env")

client = OpenAI()
MODEL = "gpt-5.4-mini"


In [2]:
chunks = json.load(open("../data/chunks.json"))
print(f"{len(chunks)} chunks from {len({c['doc_id'] for c in chunks})} documents")

589 chunks from 122 documents


## Sample chunks

Generating questions for all 589 chunks would cost more without teaching us
more. We sample 150 chunks (fixed seed, so the sample is reproducible).

In [3]:
N_CHUNKS = 150
QUESTIONS_PER_CHUNK = 3

random.seed(42)
sample = random.sample(chunks, N_CHUNKS)
sample[0]["chunk_id"]

'build/integrate-ai/understand-ai-components/agents-vs-chains#1'

## Generate questions

We use structured output (a Pydantic model) so the LLM always returns exactly
the shape we need - the same technique as in module 4.

In [4]:
class Questions(BaseModel):
    questions: list[str]


PROMPT_TEMPLATE = '''
You emulate an employee using our internal n8n documentation assistant.
Formulate {n} questions that the documentation chunk below answers.

Rules:
- complete, natural questions, as a colleague would type them in a chat
- each question must be answerable from the chunk alone
- use as few exact words from the chunk as possible (paraphrase!)

Chunk (from page "{title}", section "{section}"):

{text}
'''.strip()


def generate_questions(chunk):
    prompt = PROMPT_TEMPLATE.format(
        n=QUESTIONS_PER_CHUNK,
        title=chunk["title"],
        section=chunk["section"],
        text=chunk["text"],
    )
    response = client.responses.parse(
        model=MODEL,
        input=[{"role": "user", "content": prompt}],
        text_format=Questions,
    )
    return response.output_parsed.questions

In [5]:
generate_questions(sample[0])

['How are agents and chains different in n8n when it comes to keeping track of earlier conversation context?',
 'What role do add-on resources play when an AI model needs extra information or needs to do a specific task?',
 'Why would I choose an AI agent instead of a chain if I want a back-and-forth conversation to keep its context?']

In [6]:
rows = []

for chunk in tqdm(sample):
    for question in generate_questions(chunk):
        rows.append({
            "question": question,
            "chunk_id": chunk["chunk_id"],
            "doc_id": chunk["doc_id"],
        })

df_ground_truth = pd.DataFrame(rows)
df_ground_truth.head()

  0%|          | 0/150 [00:00<?, ?it/s]

,question,chunk_id,doc_id
0,How do AI agents differ from AI chains in n8n ...,build/integrate-ai/understand-ai-components/ag...,build/integrate-ai/understand-ai-components/ag...
1,"What’s the role of a tool in this AI setup, an...",build/integrate-ai/understand-ai-components/ag...,build/integrate-ai/understand-ai-components/ag...
2,Why would I choose an agent instead of a chain...,build/integrate-ai/understand-ai-components/ag...,build/integrate-ai/understand-ai-components/ag...
3,"Who is allowed to create custom variables, and...",build/code-in-n8n/define-custom-variables#0,build/code-in-n8n/define-custom-variables
4,What’s the difference between a global variabl...,build/code-in-n8n/define-custom-variables#0,build/code-in-n8n/define-custom-variables


In [7]:
df_ground_truth.to_csv("../data/ground-truth.csv", index=False)
len(df_ground_truth)

450

The ground truth is saved. Continue with
[2-retrieval-evaluation.ipynb](2-retrieval-evaluation.ipynb).